# Experiment 2 — XLM-R Causal Masking + Burstiness Analysis

This notebook trains an **XLM-RoBERTa** encoder with a manually applied causal mask
and then analyses whether **multitask learning** (jointly predicting switch + duration)
specifically improves recall in **bursty** code-switching regions.

## Background

XLM-R is bidirectional by design. To make predictions *anticipatory* (position t predicts
what comes at t+1), we apply an upper-triangular causal mask so each token only attends
to its left context.

## Burstiness hypothesis

A **bursty** region is one where ≥ `burst_threshold` other switch events occur within
a ±`window_size` token window. We hypothesise that the duration auxiliary task helps
the model detect these dense switching patterns better than the switch task alone.

In [ ]:
import os, sys, gc, pickle
from pathlib import Path

REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from codeswitch.auth import hf_login
from codeswitch.burstiness import (
    compute_burstiness_metrics,
    plot_burst_delta,
    plot_burstiness_comparison,
    plot_density_recall_scatter,
    print_burstiness_table,
)
from codeswitch.config import (
    ALL_LANGUAGE_PAIRS, BACKBONE_MODEL_DEFAULTS, TRAIN_PAIRS, ZEROSHOT_PAIRS,
    BurstinessConfig, DataConfig, ModelConfig, TrainConfig,
)
from codeswitch.data import build_datasets, make_collate_fn
from codeswitch.evaluate import evaluate, evaluate_per_pair, print_sigma_summary
from codeswitch.model import build_model
from codeswitch.results_json import save_results_json
from codeswitch.trainer import (
    compute_class_weights, make_warmup_cosine_scheduler, set_seed, train_epoch,
)
from codeswitch.visualize import (
    plot_grouped_f1_bars, plot_lr_schedule, plot_per_pair_training_curves,
    plot_training_history, plot_universality,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Configuration

In [ ]:
model_cfg = ModelConfig(
    backbone="xlmr",
    model_name="xlm-roberta-base",
    max_len=256,
    dropout=0.1,
    freeze_encoder=False,
)

data_cfg = DataConfig(
    dataset_name="Shelton1013/SwitchLingua_text",
    max_samples_per_pair=6000,
    train_ratio=0.8,
)

# Multitask config (lambda_dur=1.0): used for the main trained model
train_cfg_mt = TrainConfig(
    batch_size=32,
    num_epochs=16,
    base_lr=1e-5,
    head_lr_multiplier=50.0,
    warmup_ratio=0.1,
    lambda_dur=1.0,
    seed=42,
)

# Single-task config (lambda_dur=0.0): trained for burstiness comparison
train_cfg_st = TrainConfig(
    batch_size=32,
    num_epochs=16,
    base_lr=1e-5,
    head_lr_multiplier=50.0,
    warmup_ratio=0.1,
    lambda_dur=0.0,   # switch-only
    seed=42,
)

# Burstiness analysis parameters
burst_cfg = BurstinessConfig(
    burst_threshold=3,   # ≥3 other switches in ±5 window → bursty
    window_size=5,
    max_len=model_cfg.max_len,
    train_ratio=data_cfg.train_ratio,
)

train_pairs    = TRAIN_PAIRS
zeroshot_pairs = ZEROSHOT_PAIRS

CHECKPOINT_MT = "checkpoints/best_xlmr_mt.pt"
CHECKPOINT_ST = "checkpoints/best_xlmr_st.pt"
DATA_PICKLE   = "data/preprocessed.pkl"
RESULTS_JSON  = "results/train_xlmr_mt.json"

print(f"Multitask lambda_dur: {train_cfg_mt.lambda_dur}")
print(f"Single-task lambda_dur: {train_cfg_st.lambda_dur}")
print(f"Burst threshold: {burst_cfg.burst_threshold}  |  Window: ±{burst_cfg.window_size}")

## Data Loading

In [ ]:
if Path(DATA_PICKLE).exists():
    print(f"Loading cached data from {DATA_PICKLE}")
    with open(DATA_PICKLE, "rb") as f:
        all_stats = pickle.load(f)
    print(f"  Loaded {len(all_stats)} language pairs")
else:
    print("Cached data not found — running preprocessing...")
    hf_login()

    from datasets import load_dataset
    from codeswitch.data import analyze_language_pair
    from codeswitch.lid import ProductionLID

    dataset = load_dataset(data_cfg.dataset_name)
    tokenizer_pre = AutoTokenizer.from_pretrained(model_cfg.model_name)
    lid = ProductionLID(device=0 if torch.cuda.is_available() else -1)

    all_stats = {}
    for lang1, lang2 in ALL_LANGUAGE_PAIRS:
        stats = analyze_language_pair(
            dataset["train"], lang1, lang2, lid, tokenizer_pre,
            max_samples=data_cfg.max_samples_per_pair,
        )
        if stats:
            all_stats[f"{lang1}-{lang2}"] = stats

    Path(DATA_PICKLE).parent.mkdir(parents=True, exist_ok=True)
    with open(DATA_PICKLE, "wb") as f:
        pickle.dump(all_stats, f)
    print(f"✓ Saved to {DATA_PICKLE}")

## Helper: Build DataLoaders and Criteria

In [ ]:
def build_loaders(train_cfg):
    set_seed(train_cfg.seed)
    tok     = AutoTokenizer.from_pretrained(model_cfg.model_name)
    collate = make_collate_fn(tok.pad_token_id)
    train_ds, val_ds = build_datasets(
        all_stats, train_pairs, tok,
        train_ratio=data_cfg.train_ratio,
        max_len=model_cfg.max_len,
    )
    tr_ldr = DataLoader(train_ds, batch_size=train_cfg.batch_size,
                        shuffle=True,  collate_fn=collate, num_workers=train_cfg.num_workers)
    va_ldr = DataLoader(val_ds,   batch_size=train_cfg.batch_size,
                        shuffle=False, collate_fn=collate, num_workers=train_cfg.num_workers)
    sw_w, dur_w = compute_class_weights(tr_ldr)
    sw_crit  = nn.CrossEntropyLoss(weight=sw_w.to(device),  ignore_index=-100)
    dur_crit = nn.CrossEntropyLoss(weight=dur_w.to(device))
    return tok, tr_ldr, va_ldr, sw_crit, dur_crit


def train_model(train_cfg, checkpoint_path):
    if Path(checkpoint_path).exists():
        print(f"Checkpoint found at {checkpoint_path} — loading (skip training).")
        m = build_model(model_cfg).to(device)
        m.load_state_dict(torch.load(checkpoint_path, map_location=device))
        return m, []

    tok, tr_ldr, va_ldr, sw_crit, dur_crit = build_loaders(train_cfg)
    m = build_model(model_cfg).to(device)

    opt = torch.optim.AdamW(
        [
            {"params": m.encoder.parameters(),       "lr": train_cfg.base_lr},
            {"params": m.switch_head.parameters(),   "lr": train_cfg.head_lr},
            {"params": m.duration_head.parameters(), "lr": train_cfg.head_lr},
        ],
        weight_decay=train_cfg.weight_decay,
    )
    total_steps  = train_cfg.num_epochs * len(tr_ldr)
    warmup_steps = int(total_steps * train_cfg.warmup_ratio)
    sched        = make_warmup_cosine_scheduler(opt, warmup_steps, total_steps)

    history     = []
    best_sw_f1  = 0.0
    Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, train_cfg.num_epochs + 1):
        losses  = train_epoch(m, tr_ldr, opt, device, sw_crit, dur_crit, train_cfg, sched)
        metrics = evaluate(m, va_ldr, device)
        pair_f1s = {k: v["switch_f1"] for k, v in evaluate_per_pair(
            m, all_stats, train_pairs, tok, device,
            batch_size=train_cfg.batch_size, train_ratio=data_cfg.train_ratio,
            max_len=model_cfg.max_len,
        ).items()}
        history.append({
            "epoch": epoch, "loss_sw": losses["loss_sw"], "loss_dur": losses["loss_dur"],
            "switch_f1": metrics["switch_f1"], "duration_f1": metrics["duration_f1"],
            "pair_f1s": pair_f1s,
        })
        print(
            f"Epoch {epoch:02d} | "
            f"loss_sw={losses['loss_sw']:.4f}  loss_dur={losses['loss_dur']:.4f} | "
            f"sw_F1={metrics['switch_f1']:.4f}  dur_F1={metrics['duration_f1']:.4f}"
        )
        if metrics["switch_f1"] > best_sw_f1:
            best_sw_f1 = metrics["switch_f1"]
            torch.save(m.state_dict(), checkpoint_path)
            print(f"  ✓ New best: {best_sw_f1:.4f}")

    m.load_state_dict(torch.load(checkpoint_path, map_location=device))
    return m, history

## Train Multitask Model (λ=1.0)

In [ ]:
print("=" * 60)
print(f"Training MULTITASK model  (lambda_dur={train_cfg_mt.lambda_dur})")
print("=" * 60)
mt_model, mt_history = train_model(train_cfg_mt, CHECKPOINT_MT)

## Train Single-Task Model (λ=0.0) for Burstiness Comparison

In [ ]:
print("=" * 60)
print(f"Training SINGLE-TASK model  (lambda_dur={train_cfg_st.lambda_dur})")
print("=" * 60)
st_model, st_history = train_model(train_cfg_st, CHECKPOINT_ST)

## Evaluation and Universality

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_cfg.model_name)
eval_kwargs = dict(
    batch_size=train_cfg_mt.batch_size,
    train_ratio=data_cfg.train_ratio,
    max_len=model_cfg.max_len,
)
final_train = evaluate_per_pair(mt_model, all_stats, train_pairs, tokenizer, device, **eval_kwargs)
print_sigma_summary(final_train, {})

## Standard Visualizations

In [ ]:
if mt_history:
    plot_training_history(mt_history, output_path="results/xlmr_mt_training_history.png")
    pair_keys = [f"{l1}-{l2}" for l1, l2 in train_pairs]
    plot_per_pair_training_curves(
        mt_history, pair_keys, output_path="results/xlmr_mt_per_pair_curves.png"
    )

plot_universality(
    final_train, {},
    output_path="results/xlmr_universality.png",
    title="XLM-R Causal: Anticipatory F1 Across 15 Language Pairs",
)
plot_grouped_f1_bars(
    final_train,
    output_path="results/xlmr_grouped_f1.png",
    title="XLM-R Causal: Switch F1 vs Duration F1 per Pair",
)

## Burstiness Analysis

For each true switch event in the validation set, we classify it as:
- **Bursty**: ≥ `burst_threshold` other switch events within ±`window_size` tokens
- **Isolated**: fewer nearby switches

We then compare the *recall* of both the multitask (MT) and single-task (ST) models
separately for bursty and isolated events.

In [ ]:
print("Computing burstiness metrics for multitask model...")
mt_burst = compute_burstiness_metrics(
    mt_model, all_stats, train_pairs, tokenizer, device, burst_cfg
)

print("\nComputing burstiness metrics for single-task model...")
st_burst = compute_burstiness_metrics(
    st_model, all_stats, train_pairs, tokenizer, device, burst_cfg
)

In [ ]:
print_burstiness_table(mt_burst, st_burst)

In [ ]:
plot_burstiness_comparison(
    mt_burst, st_burst,
    output_path="results/burstiness_comparison.png",
)

plot_burst_delta(
    mt_burst, st_burst,
    output_path="results/burstiness_delta.png",
)

plot_density_recall_scatter(
    mt_burst, all_stats,
    output_path="results/burstiness_density_scatter.png",
)

## Save Results

In [ ]:
payload = {
    "backbone":         model_cfg.backbone,
    "model_name":       model_cfg.model_name,
    "history":          mt_history,
    "train_results":    final_train,
    "burstiness_mt":    mt_burst,
    "burstiness_st":    st_burst,
}
save_results_json(RESULTS_JSON, payload)
print(f"✓ JSON saved to {RESULTS_JSON}")